In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
df=pd.read_csv('Autismdata.csv')
df

,Age,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,Sex,Jauundice,Family_ASD,Class
0,15,1,1,0,1,0,0,1,1,0,0,m,no,no,NO
1,15,0,1,1,1,0,1,1,0,1,0,m,no,no,NO
2,15,1,1,1,0,1,1,1,1,1,1,f,no,yes,YES
3,16,1,1,1,1,1,1,1,1,0,0,f,no,no,YES
4,15,1,1,1,1,1,1,1,1,1,1,f,no,no,YES
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6070,2,0,0,0,0,0,0,0,0,0,1,f,no,yes,NO
6071,1,0,0,1,1,1,0,1,0,1,0,m,yes,no,NO
6072,2,1,0,1,1,1,1,1,1,1,1,m,yes,no,NO
6073,2,1,0,0,0,0,0,0,1,0,1,m,no,yes,NO


In [2]:
df.isnull().sum()

Age           0
A1            0
A2            0
A3            0
A4            0
A5            0
A6            0
A7            0
A8            0
A9            0
A10           0
Sex           0
Jauundice     0
Family_ASD    0
Class         0
dtype: int64

In [3]:
df.duplicated()

0       False
1       False
2       False
3       False
4       False
        ...  
6070    False
6071    False
6072    False
6073    False
6074    False
Length: 6075, dtype: bool

In [4]:
encoder=LabelEncoder()
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score,r2_score, precision_score
from sklearn.model_selection import train_test_split

In [5]:
df['Sex']=encoder.fit_transform(df['Sex'])
df['Jauundice']=encoder.fit_transform(df['Jauundice'])
df['Family_ASD']=encoder.fit_transform(df['Family_ASD'])
df['Class']=encoder.fit_transform(df['Class'])


In [6]:
x=df.iloc[:,:-1] 
#independent x
y=df.iloc[:,-1]
x
y

0       0
1       0
2       1
3       1
4       1
       ..
6070    0
6071    0
6072    0
6073    0
6074    0
Name: Class, Length: 6075, dtype: int32

In [7]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.25)
x_train

,Age,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,Sex,Jauundice,Family_ASD
1654,22,1,1,1,1,0,1,1,1,1,1,0,0,0
1788,21,0,0,0,0,0,0,0,0,0,0,0,0,1
2809,19,1,1,1,1,1,0,0,0,0,1,1,0,0
3830,47,1,0,0,0,1,0,0,0,0,1,0,0,0
1371,20,1,1,0,1,1,1,1,1,1,1,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5925,3,1,1,0,0,1,0,1,1,1,0,1,0,0
1958,52,1,0,1,1,1,0,0,1,0,0,1,0,0
503,14,1,1,1,1,1,1,0,1,1,1,1,0,0
650,38,0,1,1,1,0,0,0,0,0,0,1,0,0


In [8]:
y_train

1654    1
1788    0
2809    0
3830    0
1371    1
       ..
5925    0
1958    0
503     1
650     0
941     1
Name: Class, Length: 4556, dtype: int32

In [10]:
#model
model=RandomForestClassifier(n_estimators=50,criterion='gini')
model.fit(x_train,y_train)

RandomForestClassifier(n_estimators=50)

In [11]:
y_pred=model.predict(x_test)
y_pred

array([0, 0, 0, ..., 0, 0, 1])

In [12]:
accuracy_score(y_test,y_pred)

0.9789335088874259

In [13]:
precision_score(y_test,y_pred)

0.9823008849557522

In [14]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.98      0.99      0.98      1051
           1       0.98      0.95      0.97       468

    accuracy                           0.98      1519
   macro avg       0.98      0.97      0.98      1519
weighted avg       0.98      0.98      0.98      1519



In [15]:
dict_report=classification_report(y_test,y_pred,output_dict=True)

In [16]:
dict_report

{'0': {'precision': 0.9775070290534208,
  'recall': 0.9923882017126546,
  'f1-score': 0.9848914069877243,
  'support': 1051.0},
 '1': {'precision': 0.9823008849557522,
  'recall': 0.9487179487179487,
  'f1-score': 0.9652173913043478,
  'support': 468.0},
 'accuracy': 0.9789335088874259,
 'macro avg': {'precision': 0.9799039570045864,
  'recall': 0.9705530752153017,
  'f1-score': 0.9750543991460361,
  'support': 1519.0},
 'weighted avg': {'precision': 0.9789840037488066,
  'recall': 0.9789335088874259,
  'f1-score': 0.9788298932682904,
  'support': 1519.0}}

In [17]:
import joblib
joblib.dump(model,'autism_detection.pkl')

['autism_detection.pkl']

In [18]:
import mlflow


In [19]:
mlflow.set_experiment('Projects')
mlflow.set_tracking_uri(uri='http://127.0.0.1:5000')
with mlflow.start_run(run_name='Autism_prediction'):
    mlflow.log_metrics({
        'accuracy': dict_report['accuracy'],
        'f1_score':dict_report['weighted avg']['f1-score'],
        'recall_score': dict_report['1']['recall'],
        'support': dict_report['0']['support'],
    })
    mlflow.sklearn.log_model(model,'Random_ForestModel')

c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:140: FutureWarning: Filesystem tracking backend (e.g., './mlruns') is deprecated. Please switch to a database backend (e.g., 'sqlite:///mlflow.db'). For feedback, see: https://github.com/mlflow/mlflow/issues/18534
  return FileStore(store_uri, store_uri)
2026/01/28 16:53:18 INFO mlflow.tracking.fluent: Experiment with name 'Projects' does not exist. Creating a new experiment.


2026/01/28 16:53:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/01/28 16:53:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run Autism_prediction at: http://127.0.0.1:5000/#/experiments/688749026422565593/runs/d74ae60c7c2f4ab7bdbf887cccd43624
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/688749026422565593


In [21]:
modelname='Random_ForestModel'
run_id="d74ae60c7c2f4ab7bdbf887cccd43624"
model_uri=f"runs:/{run_id}/{modelname}"
mlflow.register_model(model_uri,modelname)

Registered model 'Random_ForestModel' already exists. Creating a new version of this model...
2026/01/28 16:55:50 WARNING mlflow.tracking._model_registry.fluent: Run with id d74ae60c7c2f4ab7bdbf887cccd43624 has no artifacts at artifact path 'Random_ForestModel', registering model based on models:/m-6f127d2bfd2c4f599774c9bfd563b50a instead
2026/01/28 16:55:50 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Random_ForestModel, version 2
Created version '2' of model 'Random_ForestModel'.


<ModelVersion: aliases=[], creation_timestamp=1769599550167, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1769599550167, metrics=None, model_id=None, name='Random_ForestModel', params=None, run_id='d74ae60c7c2f4ab7bdbf887cccd43624', run_link='', source='models:/m-6f127d2bfd2c4f599774c9bfd563b50a', status='READY', status_message=None, tags={}, user_id='', version='2'>